<a href="https://colab.research.google.com/github/maheshvlfm-coder/Forecasting-/blob/main/F1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this in Google Colab cell to install dependencies
!pip install streamlit statsmodels scikit-learn matplotlib plotly fpdf

# Save this code to a file (e.g. app.py) and run with: !streamlit run app.py --server.port 8501 --server.headless true
# To access UI in Colab, use ngrok or similar tunneling (not shown here for brevity).

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from fpdf import FPDF
import io
from datetime import datetime

st.set_page_config(page_title="Demand Forecasting App", layout="wide")


# --- Utility Functions ---

def generate_sample_data():
    """Generate sample monthly sales data with seasonality and trend."""
    periods = pd.date_range(start="2022-01-01", periods=24, freq='M')
    sales = 50 + 2*np.arange(24) + 10*np.sin(np.arange(24)*2*np.pi/12) + np.random.normal(0, 5, 24)
    df = pd.DataFrame({"period": periods, "sales": np.round(sales, 1)})
    df['period'] = df['period'].dt.to_period('M').dt.to_timestamp()
    return df

def save_sample_csv():
    df = generate_sample_data()
    csv = df.to_csv(index=False)
    return csv

def validate_data(df):
    # Validate required columns
    if not {'period', 'sales'}.issubset(df.columns):
        st.error("Uploaded CSV must contain 'period' and 'sales' columns.")
        return False
    # Parse period to datetime if not
    try:
        df['period'] = pd.to_datetime(df['period'])
    except Exception as e:
        st.error(f"Error parsing 'period' column to dates: {e}")
        return False
    # Check sales is numeric
    if not pd.api.types.is_numeric_dtype(df['sales']):
        st.error("'sales' column must be numeric.")
        return False
    return True

def plot_time_series(df):
    fig = px.line(df, x='period', y='sales', title='Sales Over Time')
    st.plotly_chart(fig, use_container_width=True)

def plot_histogram(df):
    fig = px.histogram(df, x='sales', nbins=20, title='Sales Distribution')
    st.plotly_chart(fig, use_container_width=True)

def decompose_series(df):
    from statsmodels.tsa.seasonal import seasonal_decompose
    df_indexed = df.set_index('period')
    try:
        decomposition = seasonal_decompose(df_indexed['sales'], model='additive', period=12)
        st.subheader("Seasonal Decomposition")
        fig, axs = plt.subplots(4,1, figsize=(10,8), sharex=True)
        axs[0].plot(decomposition.observed)
        axs[0].set_title('Observed')
        axs[1].plot(decomposition.trend)
        axs[1].set_title('Trend')
        axs[2].plot(decomposition.seasonal)
        axs[2].set_title('Seasonal')
        axs[3].plot(decomposition.resid)
        axs[3].set_title('Residual')
        plt.tight_layout()
        st.pyplot(fig)
    except Exception as e:
        st.warning("Seasonal decomposition could not be performed (maybe insufficient data or no clear seasonality).")

def calculate_errors(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, mape

def naive_forecast(train, test):
    # Naive forecast: last known value for all future points
    last = train.iloc[-1]
    forecast = np.repeat(last, len(test))
    return forecast

def moving_average_forecast(train, test, window=3):
    # Moving Average forecast: average of last window points as forecast for each future point
    ma = train.rolling(window=window).mean().iloc[-1]
    forecast = np.repeat(ma, len(test))
    return forecast

def arima_forecast(train, test):
    # Automatically select ARIMA order can be improved, using (1,1,1) for demo
    try:
        model = ARIMA(train, order=(1,1,1))
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=len(test))
        return forecast.values
    except Exception as e:
        st.warning(f"ARIMA model failed: {e}")
        return np.repeat(np.nan, len(test))

def exp_smoothing_forecast(train, test):
    try:
        model = ExponentialSmoothing(train, seasonal='additive', seasonal_periods=12)
        model_fit = model.fit()
        forecast = model_fit.forecast(len(test))
        return forecast.values
    except Exception as e:
        st.warning(f"Exponential smoothing model failed: {e}")
        return np.repeat(np.nan, len(test))

def random_forest_forecast(train, test):
    # Use lag features for RF regression
    df = train.to_frame(name='sales')
    for lag in range(1, 4):
        df[f'lag_{lag}'] = df['sales'].shift(lag)
    df.dropna(inplace=True)

    X_train = df.drop(columns=['sales'])
    y_train = df['sales']

    # Build test feature set similarly, based on last values from train + predicted points
    preds = []
    last_known = train.values[-3:].tolist()

    for i in range(len(test)):
        features = np.array(last_known[-3:]).reshape(1, -1)
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(features)[0]
        preds.append(pred)
        last_known.append(pred)

    return np.array(preds)

def display_performance_table(results):
    df_perf = pd.DataFrame(results).T
    df_perf.columns = ['MAE', 'RMSE', 'MAPE']
    st.subheader("Model Performance Comparison")
    st.dataframe(df_perf.style.highlight_min(axis=0))


# --- Streamlit UI ---

st.title("Demand Forecasting Application")

# Sidebar for file operations
st.sidebar.header("Upload & Sample Data")

if st.sidebar.button("Download Sample Data"):
    sample_csv = save_sample_csv()
    st.sidebar.download_button("Download CSV", sample_csv, file_name="sample_sales.csv", mime="text/csv")

uploaded_file = st.sidebar.file_uploader("Upload sales CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)

    if validate_data(df):
        st.success("Data loaded and validated successfully.")

        st.subheader("Uploaded Sales Data Snapshot")
        st.dataframe(df.head())

        # Data analysis & insights
        st.subheader("Descriptive Statistics")
        st.write(df['sales'].describe())

        plot_time_series(df)
        plot_histogram(df)
        decompose_series(df)

        # Prepare data for modeling
        df = df.sort_values('period')
        df.set_index('period', inplace=True)
        df = df.asfreq('M')
        df['sales'].interpolate(method='linear', inplace=True)

        # Train/test split - last 6 months test
        n_test = 6
        train = df['sales'][:-n_test]
        test = df['sales'][-n_test:]

        st.subheader("Forecasting Models & Results")

        models_results = {}

        # Naive
        naive_pred = naive_forecast(train, test)
        naive_mae, naive_rmse, naive_mape = calculate_errors(test, naive_pred)
        models_results['Naive'] = [naive_mae, naive_rmse, naive_mape]

        # Moving Average
        ma_pred = moving_average_forecast(train, test)
        ma_mae, ma_rmse, ma_mape = calculate_errors(test, ma_pred)
        models_results['Moving Average'] = [ma_mae, ma_rmse, ma_mape]

        # ARIMA
        arima_pred = arima_forecast(train, test)
        arima_mae, arima_rmse, arima_mape = calculate_errors(test, arima_pred)
        models_results['ARIMA'] = [arima_mae, arima_rmse, arima_mape]

        # Exponential Smoothing
        exp_pred = exp_smoothing_forecast(train, test)
        exp_mae, exp_rmse, exp_mape = calculate_errors(test, exp_pred)
        models_results['Exp Smoothing'] = [exp_mae, exp_rmse, exp_mape]

        # Random Forest
        rf_pred = random_forest_forecast(train, test)
        rf_mae, rf_rmse, rf_mape = calculate_errors(test, rf_pred)
        models_results['Random Forest'] = [rf_mae, rf_rmse, rf_mape]

        # Show performance
        display_performance_table(models_results)

        best_model = min(models_results, key=lambda k: models_results[k][1])  # best RMSE
        st.markdown(f"### Best Model Based on RMSE: **{best_model}**")

        # Visualize best model prediction vs actual
        fig = px.line(title=f"Actual vs Forecast - {best_model}")
        fig.add_scatter(x=test.index, y=test.values, mode='lines+markers', name='Actual')

        if best_model == 'Naive':
            fig.add_scatter(x=test.index, y=naive_pred, mode='lines+markers', name='Forecast')
        elif best_model == 'Moving Average':
            fig.add_scatter(x=test.index, y=ma_pred, mode='lines+markers', name='Forecast')
        elif best_model == 'ARIMA':
            fig.add_scatter(x=test.index, y=arima_pred, mode='lines+markers', name='Forecast')
        elif best_model == 'Exp Smoothing':
            fig.add_scatter(x=test.index, y=exp_pred, mode='lines+markers', name='Forecast')
        else:
            fig.add_scatter(x=test.index, y=rf_pred, mode='lines+markers', name='Forecast')

        st.plotly_chart(fig, use_container_width=True)

        # Download forecast CSV
        forecast_df = pd.DataFrame({
            'period': test.index,
            'actual_sales': test.values,
            'forecasted_sales': (naive_pred if best_model == 'Naive' else
                                 ma_pred if best_model == 'Moving Average' else
                                 arima_pred if best_model == 'ARIMA' else
                                 exp_pred if best_model == 'Exp Smoothing' else
                                 rf_pred)
        })
        csv_forecast = forecast_df.to_csv(index=False)
        st.download_button("Download Forecast Results", csv_forecast, file_name="forecast_results.csv", mime="text/csv")

        # Download comparison results CSV
        comparison_df = pd.DataFrame(models_results).T
        comparison_df.columns = ['MAE', 'RMSE', 'MAPE']
        csv_comparison = comparison_df.to_csv()
        st.download_button("Download Model Comparison", csv_comparison, file_name="model_comparison.csv", mime="text/csv")

else:
    st.info("Please upload a CSV file with columns `period` and `sales`, or download the sample data to get started.")

st.markdown("""
---
#### Help / About
This app supports demand forecasting using multiple methods to find the best fit:
- Naive: last value carried forward.
- Moving Average: average of recent points.
- ARIMA: classic time series model with trend and auto-regression.
- Exponential Smoothing: captures level, trend, seasonality.
- Random Forest: machine learning with lag features.

Upload your monthly sales data with 'period' and 'sales' columns to get started.

Developed for interactive forecasting analytics in supply chain and business.
""")
